# Challenge 01: Diagonal Unitary Circuit Composer

Write a program that takes as an input a diagonal unitary matrix, and returns a corresponding quantum circuit. For example, given the unitary matrix $\Lambda \in \mathbb{R}^{4\times4}$ shown below, the composer would synthesize the following circuit:

<img src="../images/composer_cir.png" width = 750/>

For this problem, you are not restricted to the use of a particular set of gates. Since different equivalent circuits can have the same unitary operator associated with them, your circuit for $\Lambda$ wouldn't have to necessarily look the same as the one above. For example, depending on your choice of gates, the same unitary $\Lambda$ above could result in circuits with more (or less) number of gates, like the ones shown below:

<img src="../images/composer_cir2.png" width = 680/>

We have broken down this challenge in three levels of difficulty. You don't have to work on these separately nor complete all three of them; If you have an idea how to tackle Level 3, you can just go for it, but if not, solving first two first should help you get started.

**Level 1:** The composer should return a circuit for any diagonal unitary matrix $\Lambda \in \mathbb{R}^{4\times4}$. (i.e., $4 \times 4$ matrices with only real entries)

**Level 2:** The composer should return a circuit for any diagonal unitary matrix $\Lambda \in \mathbb{R}^{N\times N}$. (i.e., matrices of arbitrary size $N = 2^n$ with only real entries)

**Level 3:** The composer should return a circuit for any diagonal unitary matrix $\Lambda \in \mathbb{C}^{N\times N}$. (i.e., matrices of arbitrary size $N = 2^n$ with complex entries)

**Bonus:** The composer should return circuits consisting of only one-qubit and two-qubit gates. The bonus applies to solutions in levels 2 and 3.

The idea behind this challenge is for you to develop an understanding of how circuits can be built out of unitaries, so avoid using functions that do this for you automatically, like the `transpile` function in Qiskit, or the `decompose` function in Cirq.

**Hint:** If you have trouble getting started, it might be helpful to know that this problem is closely related to the design of Oracles for Grover's Algorithm [1].

[1] Figgatt, Caroline, et al. "Complete 3-Qubit Grover search on a programmable quantum computer." Nature communications 8.1 (2017): 1-9. [arXiv:1703.10535](https://arxiv.org/pdf/1703.10535.pdf)

### Generate test matrices

In [1]:
import itertools
import numpy as np
import random
from random import Random
from qiskit import transpile
from qiskit_aer import AerSimulator

In [2]:
class RandomDiagUnitaryGenerator:
    """
    Generates random diagonal unitary matrices by sampling theta uniformly from [0, 2*np.pi)
    """

    def __init__(self, seed=42):
        self.rng = Random(seed)

    def generate_unitary(self, size):
        """size is num_rows ( = num_cols)"""
        unitary = np.zeros((size, size), dtype=complex)
        for i in range(size):
            random_angle = 2 * np.pi * self.rng.random()
            complex_num = np.exp(1j * random_angle)
            unitary[i][i] = complex_num
        return unitary

In [3]:
class ListProductDiagUnitaryGenerator:
    def __init__(self, size, angles=None):
        self.size = size
        self.angles = angles
        if self.angles is None:
            self.angles = [
                0, np.pi/2, np.pi, 3*np.pi/2
            ]
        self.num_total_unitaries = len(self.angles)**self.size
        self.prod_iter = itertools.product(self.angles, repeat=self.size)

    def generate_unitary(self):
        try:
            angles = next(self.prod_iter)
            unitary = np.zeros((self.size, self.size), dtype=complex)
            for i, ang in enumerate(angles):
                complex_num = np.exp(1j*ang)
                unitary[i][i] = complex_num
            return unitary
        except StopIteration:
            return None

In [4]:
class CircuitChecker:
    def __init__(self):
        self.simulator = AerSimulator(method="unitary")

    def ckt_correct(self, qc, tgt_unitary):
        qc.save_unitary()
        qc = transpile(qc, self.simulator)

        result = self.simulator.run(qc).result()
        ckt_unitary = result.get_unitary(qc)

        return np.allclose(ckt_unitary, tgt_unitary)

In [5]:
def run_list_prod_diag_unitary_tests(SolutionCls, size, num_unitaries=25):
    tick = "\u2714"
    wrong = "\u274c"
    ckt_checker = CircuitChecker()

    list_product_diag_unitary_generator = ListProductDiagUnitaryGenerator(size)
    num_list_product_unitaries_proc = 0
    num_total_unitaries = list_product_diag_unitary_generator.num_total_unitaries
    process_probability = min(1, num_unitaries / num_total_unitaries)

    while num_list_product_unitaries_proc < num_unitaries:
        tgt_unitary = list_product_diag_unitary_generator.generate_unitary()
        if tgt_unitary is None:
            break

        if random.random() > process_probability:
            continue

        qc = SolutionCls().solve(tgt_unitary)
        if not ckt_checker.ckt_correct(qc, tgt_unitary):
            print(f"{wrong} Failed list product diagonal unitaries test")
            return tgt_unitary
        num_list_product_unitaries_proc += 1

    print(f"{tick} {num_list_product_unitaries_proc} list product diagonal unitaries generated correctly")


def run_random_diag_unitary_tests(SolutionCls, size, num_unitaries=25):
    tick = "\u2714"
    wrong = "\u274c"
    ckt_checker = CircuitChecker()

    random_diag_unitary_generator = RandomDiagUnitaryGenerator()
    for i in range(num_unitaries):
        tgt_unitary = random_diag_unitary_generator.generate_unitary(size)

        qc = SolutionCls().solve(tgt_unitary)
        if not ckt_checker.ckt_correct(qc, tgt_unitary):
            print(f"{wrong} Failed random diagonal unitaries test")
            return tgt_unitary

    print(f"{tick} {num_unitaries} random diagonal unitaries generated correctly")


def run_tests(SolutionCls, size_list, first_phase_0=False, num_unitaries_per_size=25):
    tick = "\u2714"

    for size in size_list:
        print("-"*32 + f"Evaluating for shape = ({size}, {size})" + "-"*32)

        if size <= 8:
            res = run_list_prod_diag_unitary_tests(SolutionCls, size, num_unitaries_per_size)
            if res is not None:
                return res

        res = run_random_diag_unitary_tests(SolutionCls, size, num_unitaries_per_size)
        if res is not None:
            return res
        
        print("-"*32 + f"All tests passed for shape = ({size}, {size})" + "-"*32)

    print(f"{tick} All tests passed successfully")

### Solution 1 - With Multi Controlled Phase Gates

Let's consider a unitary matrix:

\begin{bmatrix}
e^{i\theta_0} & 0 & 0 & 0 \\
0 & e^{i\theta_1} & 0 & 0 \\
0 & 0 & e^{i\theta_2} & 0 \\
0 & 0 & 0 & e^{i\theta_3}
\end{bmatrix}

Let me represent this with the vector: [$\theta_0$, $\theta_1$, $\theta_2$, $\theta_3$]. As I apply gates, I will try to turn this vector into the all zeros vector, reaching the required unitary.

We want:
- $|00>$ -> $e^{i\theta_0}|00>$,
- $|01>$ -> $e^{i\theta_1}|01>$,
- $|10>$ -> $e^{i\theta_2}|10>$,
- $|11>$ -> $e^{i\theta_3}|11>$

If we first apply to the MSB: an $X$ gate, then a $P(\theta_0)$ gate, and then again an $X$ gate, we will have:
- $|00>$ -> $|10>$ -> $e^{i\theta_0}|10>$ -> $e^{i\theta_0}|00>$,
- $|01>$ -> $|11>$ -> $e^{i\theta_0}|11>$ -> $e^{i\theta_0}|01>$,
- $|10>$ -> $|00>$ -> $|00>$ -> $|10>$,
- $|11>$ -> $|01>$ -> $|01>$ -> $|11>$

Remaining phase to be introduced = [$0$, $\theta_1 - \theta_0$, $\theta_2$, $\theta_3$]. Let's call this vector our new tgt vector: [$0$, $\theta_1$, $\theta_2$, $\theta_3$] (just reusing names, to avoid a lot of clutter).
Now we want:
- $|00>$ -> $|00>$,
- $|01>$ -> $e^{i\theta_1}|01>$,
- $|10>$ -> $e^{i\theta_2}|10>$,
- $|11>$ -> $e^{i\theta_3}|11>$

Target unitary now:
\begin{bmatrix}
1 = e^{i.0} & 0 & 0 & 0 \\
0 & e^{i\theta_1} & 0 & 0 \\
0 & 0 & e^{i\theta_2} & 0 \\
0 & 0 & 0 & e^{i\theta_3}
\end{bmatrix}

From here, we will break this into two unitaries with similar structure (top left element = 1)

Now, if we apply a $P(\theta_2)$ gate to the MSB, we get:
- $|00>$ -> $|00>$,
- $|01>$ -> $|01>$,
- $|10>$ -> $e^{i\theta_2}|10>$,
- $|11>$ -> $e^{i\theta_2}|11>$

Remaining phase to be introduced = [$0$, $\theta_1$, $0$, $\theta_3 - \theta_2$]. Target matrix now:
\begin{bmatrix}
1 = e^{i.0} & 0 & 0 & 0 \\
0 & e^{i\theta_1} & 0 & 0 \\
0 & 0 & 1 = e^{i.0} & 0 \\
0 & 0 & 0 & e^{i(\theta_3 - \theta_2)}
\end{bmatrix}

We now have two submatrices, of half the size of our original matrix. We can get the gates for each submatrix independently and apply those gates to the LSB with an appropriate control on the MSB. We can do this recursively to get circuits for larger matrices.

In [6]:
import numpy as np
from typing import List
from qiskit import QuantumCircuit

In [7]:
class Solution1:
    """
    Solution using multi controlled phase gates
    """
    def __init__(self):
        num_qubits = -1
        self.orig_theta_vec = []
        self.gate_list = []
        self.qc = None

    def get_gates(self, control_seq: List[int], vector_to_fill: List[float]):
        n = len(vector_to_fill)
        if(n <= 1):
            return

        idx_to_make_0 = n // 2
        angle = vector_to_fill[idx_to_make_0]

        if(angle != 0):
            self.gate_list.append(
                (control_seq, angle)
            )

        for i in range(idx_to_make_0, n):
            vector_to_fill[i] -= angle

        self.get_gates(control_seq+[0], vector_to_fill[:idx_to_make_0])
        self.get_gates(control_seq+[1], vector_to_fill[idx_to_make_0:])
    
    def create_circuit(self):
        """Remember that qubit is the least significant"""
        self.qc = QuantumCircuit(self.num_qubits)

        # First make the |00...0><00...0| angle 0
        self.qc.x(self.num_qubits - 1)
        self.qc.p(self.orig_theta_vec[0], self.num_qubits - 1)
        self.qc.x(self.num_qubits - 1)

        for control_seq, angle in self.gate_list:
            qubit_to_be_applied_on = self.num_qubits  - 1 - len(control_seq)

            for i, val in enumerate(control_seq):
                if val == 0:
                    self.qc.x(self.num_qubits - i - 1)

            if qubit_to_be_applied_on == self.num_qubits - 1:  # most significant
                self.qc.p(angle, qubit_to_be_applied_on)
            else:
                self.qc.mcp(
                    angle,
                    [qubit_to_be_applied_on + i + 1 for i in range(len(control_seq))],
                    qubit_to_be_applied_on
                )

            for i, val in enumerate(control_seq):
                if val == 0:
                    self.qc.x(self.num_qubits - i - 1)

    def solve(self, unitary: np.ndarray):
        n = unitary.shape[0]
        self.num_qubits = round(np.log2(n))
        theta_vec = []
        for i in range(n):
            angle = np.angle(unitary[i][i], deg=False)
            theta_vec.append(angle)
        self.orig_theta_vec = theta_vec.copy()
        theta_0 = theta_vec[0]
        for i in range(0, 2**(self.num_qubits - 1)):
            theta_vec[i] -= theta_0
        self.get_gates([], theta_vec)
        self.create_circuit()

        return self.qc

In [8]:
test_unitary = [
    [np.exp(1j*np.pi/4), 0, 0, 0],
    [0, np.exp(1j*np.pi/4), 0, 0],
    [0, 0, np.exp(1j*np.pi/4), 0],
    [0, 0, 0, np.exp(1j*np.pi/4)],
]
test_unitary = np.array(test_unitary)

sol = Solution1()
qc = sol.solve(test_unitary)
CircuitChecker().ckt_correct(qc, test_unitary)

True

In [9]:
run_tests(
    Solution1,
    size_list=[2**(i+1) for i in range(5)],
    num_unitaries_per_size=25,
    first_phase_0=False
)

--------------------------------Evaluating for shape = (2, 2)--------------------------------
✔ 16 list product diagonal unitaries generated correctly
✔ 25 random diagonal unitaries generated correctly
--------------------------------All tests passed for shape = (2, 2)--------------------------------
--------------------------------Evaluating for shape = (4, 4)--------------------------------
✔ 25 list product diagonal unitaries generated correctly
✔ 25 random diagonal unitaries generated correctly
--------------------------------All tests passed for shape = (4, 4)--------------------------------
--------------------------------Evaluating for shape = (8, 8)--------------------------------
✔ 25 list product diagonal unitaries generated correctly
✔ 25 random diagonal unitaries generated correctly
--------------------------------All tests passed for shape = (8, 8)--------------------------------
--------------------------------Evaluating for shape = (16, 16)-------------------------------